# Chapter 9 — Escalation and Human-in-the-Loop · Practice

**Goal:** build every layer of an escalation system by hand: the escalation criteria and how to score them, ask-don't-guess tool results, deterministic gates, human approval checkpoints on each Claude surface, structured handoffs, confidence calibration, stratified audits and review routing. You finish with a small support agent that hands off cleanly and a review plan for an extraction pipeline.

- Theory: [README.md](README.md) (read section 9.x before running the matching section here)
- Estimated time: 3–4 hours
- Estimated API cost: **< $0.25** with the default model (`claude-sonnet-5-5`, every live cell once). Every section has **offline** cells that use a scripted fake API or plain Python, so you can learn all of it with no key and no credit.
- Prerequisites: [00-prerequisites](../../00-prerequisites/README.md) (venv from `requirements.txt`, `.env` with `ANTHROPIC_API_KEY`), [Chapter 2](../02-tool-use/README.md) (the tool loop, `is_error`, strict tools), [Chapter 3](../03-agent-sdk/README.md) (hooks), [Chapter 6](../06-prompt-engineering/README.md) (few-shot, validation)

How to read the cells:

| Marker | Meaning |
|---|---|
| **(offline)** | Runs without a key. Uses `ScriptedClaude`, a real `anthropic.Anthropic` client whose HTTP layer replays canned responses and records what you sent, or plain Python. |
| **(live)** | Calls the real API and costs a fraction of a cent. Without a key it prints `skipped (live)`, and no offline cell depends on a live one. |

> This notebook never starts the Claude Agent SDK or the `claude` CLI. Agent SDK hook callbacks are plain `async` functions, so section 9.5 unit-tests them directly, and the `ClaudeAgentOptions` wiring is shown as text.

In [ ]:
import json
import math
import os
import random
import secrets
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from itertools import count
from typing import Literal

import anthropic
import httpx2
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, ConfigDict, Field, ValidationError

load_dotenv(find_dotenv())

MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY; only the (live) cells use it
print("anthropic", anthropic.__version__, "| MODEL =", MODEL)

In [ ]:
# Live cells need a key. Without one they print "skipped (live)" and every (offline) cell still runs.
HAS_KEY = bool(os.getenv("ANTHROPIC_API_KEY"))
print("live cells enabled" if HAS_KEY else "no key: live cells will be skipped")

### Offline toolkit (run this once)

`ScriptedClaude` gives you a **real** SDK client (`.client`) whose transport answers with the responses you script, in order, and stores every request body in `.requests`. The SDK's own parsing, types and `messages.parse()` logic run for real; only the model is fake. `text()`, `tool_call()` and `reply()` build response JSON in the Messages API wire format. `run_coro()` runs an `async` hook callback that never really waits, which lets us test Agent SDK hooks without starting the SDK.

In [ ]:
_ids = count(1)


def text(t: str) -> dict:
    return {"type": "text", "text": t}


def tool_call(name: str, tool_input: dict, id: str | None = None) -> dict:
    return {"type": "tool_use", "id": id or f"toolu_{next(_ids):04d}", "name": name, "input": tool_input}


def reply(*blocks: dict, stop_reason: str | None = None) -> dict:
    """A Messages API response body. stop_reason defaults to tool_use/end_turn based on the blocks."""
    if stop_reason is None:
        stop_reason = "tool_use" if any(b["type"] == "tool_use" for b in blocks) else "end_turn"
    return {
        "id": f"msg_{next(_ids):04d}", "type": "message", "role": "assistant", "model": "scripted",
        "content": list(blocks), "stop_reason": stop_reason, "stop_sequence": None,
        "usage": {"input_tokens": 0, "output_tokens": 0},
    }


class ScriptedClaude:
    """Offline stand-in for the Messages API: replays scripted replies, records every request."""

    def __init__(self, *replies: dict):
        self.replies = list(replies)
        self.requests: list[dict] = []
        transport = httpx2.MockTransport(self._handle)
        self.client = anthropic.Anthropic(
            api_key="offline", max_retries=0, http_client=anthropic.DefaultHttpxClient(transport=transport)
        )

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        self.requests.append(json.loads(request.content))
        if not self.replies:
            return httpx2.Response(500, json={"type": "error", "error": {"type": "api_error", "message": "script exhausted"}})
        body = dict(self.replies.pop(0))
        return httpx2.Response(body.pop("_status", 200), json=body)


def run_coro(coro):
    """Run a coroutine that never suspends (like the hook callbacks in 9.5) without an event loop.
    Jupyter already runs a loop, so asyncio.run() would fail here."""
    try:
        coro.send(None)
    except StopIteration as done:
        return done.value
    raise RuntimeError("the coroutine suspended; await it inside a real async app instead")


print("offline toolkit ready")

### The running example: Acme Home support

Every section uses the same in-memory backend: four customers (three of them called **Maria Lopez**), four orders, and the backend functions behind the README's tools. Amounts are in USD, as in the README. `REFUND_LIMIT_USD` is the agent's authority: anything above it needs a human.

In [ ]:
CUSTOMERS = [
    {"customer_id": "CUST-48210", "name": "Ana Silva", "email": "ana@example.com", "phone": "+1-555-0101",
     "order_ids": ["ORD-31544"]},
    {"customer_id": "CUST-20001", "name": "Maria Lopez", "email": "maria.lopez@example.com", "phone": "+1-555-0110",
     "order_ids": ["ORD-70001"]},
    {"customer_id": "CUST-20002", "name": "Maria Lopez", "email": "mlopez@example.org", "phone": "+1-555-0111",
     "order_ids": ["ORD-70002"]},
    {"customer_id": "CUST-20003", "name": "Maria Lopez", "email": "maria.l@example.net", "phone": "+1-555-0112",
     "order_ids": ["ORD-70003"]},
]
ORDERS = {
    "ORD-31544": {"order_id": "ORD-31544", "customer_id": "CUST-48210", "item": "Blender", "status": "delivered",
                  "delivered_on": "2026-09-28", "total": 179.99},
    "ORD-70001": {"order_id": "ORD-70001", "customer_id": "CUST-20001", "item": "Kettle", "status": "delivered",
                  "delivered_on": "2026-09-30", "total": 89.99},
    "ORD-70002": {"order_id": "ORD-70002", "customer_id": "CUST-20002", "item": "Fridge", "status": "delivered",
                  "delivered_on": "2026-09-21", "total": 1249.00},
    "ORD-70003": {"order_id": "ORD-70003", "customer_id": "CUST-20003", "item": "Toaster", "status": "delivered",
                  "delivered_on": "2026-08-02", "total": 39.50},
}
REFUND_LIMIT_USD = 500.00
REFUNDS: list[dict] = []   # what process_refund actually paid out
TICKETS: list[dict] = []   # the human support queue


def crm_search(name: str | None = None, email: str | None = None, phone: str | None = None) -> list[dict]:
    if not any((name, email, phone)):
        return []
    return [c for c in CUSTOMERS
            if (name is None or c["name"].lower() == name.lower())
            and (email is None or c["email"].lower() == email.lower())
            and (phone is None or c["phone"] == phone)]


def get_customer(name: str | None = None, email: str | None = None, phone: str | None = None) -> dict:
    """README 9.3: one match -> found; several -> ask for an identifier, without leaking anyone's data."""
    matches = crm_search(name=name, email=email, phone=phone)
    if len(matches) == 1:
        c = matches[0]
        return {"status": "found", "customer_id": c["customer_id"], "name": c["name"], "order_ids": c["order_ids"]}
    if len(matches) > 1:
        return {"status": "multiple_matches", "count": len(matches), "ask_for_one_of": ["email", "phone", "order_number"]}
    return {"status": "not_found", "hint": "Ask the customer to check the spelling or give their email."}


def lookup_order(order_id: str) -> dict:
    order = ORDERS.get(order_id)
    return dict(order) if order else {"status": "not_found", "hint": "Check the order number with the customer."}


def process_refund(order_id: str, amount: float, reason: str) -> dict:
    refund_id = f"RF-{len(REFUNDS) + 1:04d}"
    REFUNDS.append({"refund_id": refund_id, "order_id": order_id, "amount": amount, "reason": reason})
    return {"status": "refunded", "refund_id": refund_id, "order_id": order_id, "amount": amount}


def file_ticket(summary: dict) -> dict:
    ticket_id = f"HT-{len(TICKETS) + 1:04d}"
    TICKETS.append({"ticket_id": ticket_id, **summary})
    return {"status": "queued", "ticket_id": ticket_id, "eta": "within 4 business hours"}


def reset_backend() -> None:
    REFUNDS.clear()
    TICKETS.clear()


def nullable(t: str) -> dict:
    return {"type": [t, "null"]}


SUPPORT_TOOLS = [
    {
        "name": "get_customer",
        "description": (
            "Find the customer's account by name, email or phone (pass null for the ones you don't have). "
            "Returns status 'found' with one customer_id, 'multiple_matches' (then ask the customer for one of "
            "the identifiers listed in ask_for_one_of), or 'not_found'. Call it first: lookup_order and "
            "process_refund only work after it has returned exactly one match."
        ),
        "strict": True,
        "input_schema": {"type": "object", "additionalProperties": False,
                         "properties": {"name": nullable("string"), "email": nullable("string"), "phone": nullable("string")},
                         "required": ["name", "email", "phone"]},
    },
    {
        "name": "lookup_order",
        "description": "Fetch one order (item, status, delivery date, total in USD) by its ID, format ORD-#####. "
                       "Only works for the verified customer's orders.",
        "strict": True,
        "input_schema": {"type": "object", "additionalProperties": False,
                         "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]},
    },
    {
        "name": "process_refund",
        "description": "Refund an amount in USD on an order that lookup_order returned for the verified customer. "
                       "Refunds above the agent's limit are blocked and must be escalated.",
        "strict": True,
        "input_schema": {"type": "object", "additionalProperties": False,
                         "properties": {"order_id": {"type": "string"}, "amount": {"type": "number"},
                                        "reason": {"type": "string"}},
                         "required": ["order_id", "amount", "reason"]},
    },
]
print(get_customer(email="ana@example.com"))
print(get_customer(name="Maria Lopez"))

## 9.1 When to escalate

The exam's three appropriate triggers are an **explicit request for a human**, a **policy gap or exception**, and **inability to make progress**; code adds deterministic **business limits**. Negative sentiment, complexity on its own and the model's self-rated confidence are *not* triggers. An escalation policy is a classifier, so you measure it on labeled cases. [Theory](README.md#91-when-to-escalate)

### A labeled set of customer turns (offline)

Fourteen turns with the decision a well-calibrated agent makes: `resolve`, `ask` (a clarifying question the customer can answer) or `escalate` with a reason. `flags` hold facts the agent would know from tool results or the conversation so far. `self_conf` is the 1–10 confidence a model *might* report if you asked it. Those numbers are illustrative, chosen to show the failure mode.

In [ ]:
REASONS = ["customer_requested_human", "policy_gap", "cannot_make_progress", "threshold_exceeded", "other"]

LABELED_TURNS = [
    {"id": "t01", "message": "I want to speak to a manager.",
     "context": "First message of the conversation. No lookups yet.",
     "flags": {"asks_human": True}, "self_conf": 9, "expected": ("escalate", "customer_requested_human")},
    {"id": "t02", "message": "My blender broke after 3 days, this is ridiculous!! Worst purchase ever.",
     "context": "Verified customer CUST-48210. Order ORD-31544 is inside the 30-day replacement window.",
     "flags": {}, "self_conf": 4, "expected": ("resolve", None)},
    {"id": "t03", "message": "Another shop lists this kettle for $25 less. Can you beat that?",
     "context": "Verified. Policy: we refund the difference if OUR price drops within 10 days of delivery. Other retailers are not mentioned.",
     "flags": {"policy_covers": None}, "self_conf": 8, "expected": ("escalate", "policy_gap")},
    {"id": "t04", "message": "The parcel arrived crushed, photo attached. I'd like a new one.",
     "context": "Verified. Damage with photo evidence is covered by the replacement policy.",
     "flags": {}, "self_conf": 5, "expected": ("resolve", None)},
    {"id": "t05", "message": "Hi, this is Maria Lopez, please refund my last order.",
     "context": "get_customer(name='Maria Lopez') returned multiple_matches (count 3). No identifier requested yet.",
     "flags": {"matches": 3}, "self_conf": 7, "expected": ("ask", None)},
    {"id": "t06", "message": "I don't want a replacement. Let me talk to a real person.",
     "context": "The agent already acknowledged the problem and offered a replacement or a refund.",
     "flags": {"asks_human": True}, "self_conf": 6, "expected": ("escalate", "customer_requested_human")},
    {"id": "t07", "message": "I've been a customer for ten years. Please waive the restocking fee.",
     "context": "Verified. Policy: opened items carry a 15% restocking fee. No exception process is defined.",
     "flags": {"policy_covers": None}, "self_conf": 8, "expected": ("escalate", "policy_gap")},
    {"id": "t08", "message": "My fridge died after a week. Refund the full $1,249.",
     "context": "Verified. Covered by warranty, but process_refund returned: BLOCKED by policy, refunds over $500 need human approval.",
     "flags": {"gate_blocked": True}, "self_conf": 9, "expected": ("escalate", "threshold_exceeded")},
    {"id": "t09", "message": "I already told you, I don't remember which email I used!",
     "context": "get_customer returned not_found twice. The agent has asked for an identifier twice.",
     "flags": {"matches": 0, "id_requests": 2}, "self_conf": 3, "expected": ("escalate", "cannot_make_progress")},
    {"id": "t10", "message": "Third time writing about this. My order is 9 days late, sort it out!",
     "context": "Verified. Policy: refund the shipping fee when delivery is more than 7 days late.",
     "flags": {}, "self_conf": 3, "expected": ("resolve", None)},
    {"id": "t11", "message": "Quick one: I'd like to return an unopened toaster I bought 10 days ago.",
     "context": "Verified. Policy: unopened items can be returned within 30 days.",
     "flags": {}, "self_conf": 9, "expected": ("resolve", None)},
    {"id": "t12", "message": "Where is my order ORD-70001?",
     "context": "The order number and the sender's email identify exactly one customer (CUST-20001).",
     "flags": {"matches": 1}, "self_conf": 8, "expected": ("resolve", None)},
    {"id": "t13", "message": "Can you check why my order status hasn't changed?",
     "context": "Verified. lookup_order failed 3 times (backend timeout) after its own retries.",
     "flags": {"tool_failures": 3}, "self_conf": 6, "expected": ("escalate", "cannot_make_progress")},
    {"id": "t14", "message": "Your website is so confusing. How do I change the delivery address for ORD-31544?",
     "context": "Verified. The order has not shipped; address changes are allowed before shipping.",
     "flags": {}, "self_conf": 4, "expected": ("resolve", None)},
]
print(Counter(c["expected"][0] for c in LABELED_TURNS))

### Three routers, one scorecard (offline)

- `sentiment_router`: escalate when a keyword sentiment score drops to −0.6 or below (the teammate's proposal from the README self-check).
- `confidence_router`: escalate when the model rates itself below 6/10.
- `criteria_router`: the explicit criteria from the README, applied to the facts in `flags`, in the rule order of README 9.1–9.3.

`score_decisions` reports what you would report to stakeholders: decision accuracy, escalation **precision** (were escalations necessary?), escalation **recall** (did every case that needed a human get one?), **reason accuracy** and the share of resolvable cases the agent **kept**.

In [ ]:
NEGATIVE_CUES = ("ridiculous", "worst", "outrageous", "terrible", "sort it out", "!!", "already told you",
                 "confusing", "third time")


def sentiment(message: str) -> float:
    hits = sum(cue in message.lower() for cue in NEGATIVE_CUES)
    return max(-1.0, -0.35 * hits)


def sentiment_router(case: dict) -> tuple[str, str | None]:
    return ("escalate", "other") if sentiment(case["message"]) <= -0.6 else ("resolve", None)


def confidence_router(case: dict) -> tuple[str, str | None]:
    return ("escalate", "other") if case["self_conf"] < 6 else ("resolve", None)


def criteria_router(case: dict) -> tuple[str, str | None]:
    f = case["flags"]
    if f.get("asks_human"):
        return ("escalate", "customer_requested_human")          # honor it immediately
    if f.get("matches", 1) != 1:
        if f.get("id_requests", 0) >= 2:
            return ("escalate", "cannot_make_progress")          # asked twice, still no single match
        return ("ask", None)                                     # the customer can resolve this
    if f.get("policy_covers", True) is not True:
        return ("escalate", "policy_gap")                        # silent or ambiguous policy
    if f.get("gate_blocked"):
        return ("escalate", "threshold_exceeded")                # code said: needs a human
    if f.get("tool_failures", 0) >= 3 or f.get("rejected_offers", 0) >= 2:
        return ("escalate", "cannot_make_progress")
    return ("resolve", None)


def score_decisions(preds: list[tuple], cases: list[dict]) -> dict:
    exp = [c["expected"] for c in cases]
    should = [e[0] == "escalate" for e in exp]
    did = [p[0] == "escalate" for p in preds]
    tp = sum(s and d for s, d in zip(should, did))
    reason_ok = sum(s and d and p[1] == e[1] for s, d, p, e in zip(should, did, preds, exp))
    resolvable = [not s for s in should]
    return {
        "decision_accuracy": sum(p[0] == e[0] for p, e in zip(preds, exp)) / len(cases),
        "escalation_precision": tp / sum(did) if sum(did) else 0.0,
        "escalation_recall": tp / sum(should) if sum(should) else 0.0,
        "reason_accuracy": reason_ok / tp if tp else 0.0,
        "kept_resolvable": sum(r and not d for r, d in zip(resolvable, did)) / sum(resolvable),
    }


ROUTERS = {"sentiment": sentiment_router, "self-confidence": confidence_router, "criteria": criteria_router}
preds = {name: [router(c) for c in LABELED_TURNS] for name, router in ROUTERS.items()}

print(f"{'turn':5} {'expected':34}" + "".join(f"{n:>18}" for n in ROUTERS))
for i, c in enumerate(LABELED_TURNS):
    exp = c["expected"]
    cells = "".join(f"{('✓ ' if preds[n][i][0] == exp[0] else '✗ ') + preds[n][i][0]:>18}" for n in ROUTERS)
    print(f"{c['id']:5} {exp[0] + (':' + exp[1] if exp[1] else ''):34}{cells}")
print()
for name in ROUTERS:
    s = score_decisions(preds[name], LABELED_TURNS)
    print(f"{name:16} " + "  ".join(f"{k}={v:.0%}" for k, v in s.items()))

> **Observe:** the sentiment router escalates the angry-but-covered blender and late-delivery cases and misses the calm price-match and fee-waiver requests: mood is not complexity. The self-confidence router misses the explicit manager request (the model was "9/10 sure" it could help) and escalates easy cases it felt unsure about. It misroutes in both directions, which is exactly why the README calls it a poor trigger.

> **Try it:** add a turn where a calm customer politely asks "Could a person look at this, please?" Which routers get it right? Then add a turn with `"rejected_offers": 2` and check that only the criteria router escalates it as `cannot_make_progress`.

## 9.2 Escalation patterns

Four conversational patterns: **immediate escalation** on an explicit request, **resolve first** when the issue is in scope, **acknowledge → offer → escalate on reiteration** for frustrated customers, and **policy-gap escalation** when the policy is silent. In the prompt they become categorical criteria plus a few boundary examples. [Theory](README.md#92-escalation-patterns)

### The four patterns as a reference state machine (offline)

A deterministic version of the patterns, so you can see the exact action sequence each conversation should produce. A real agent gets this behavior from the prompt; the reference is what your evals compare against.

In [ ]:
@dataclass
class CustomerTurn:
    text: str
    asks_human: bool = False          # an explicit request for a person
    in_policy: bool | None = True     # False / None = policy is silent or ambiguous for this request
    frustrated: bool = False
    accepts_offer: bool = False
    rejects_offer: bool = False


def pattern_agent(turns: list[CustomerTurn]) -> list[str]:
    actions, rejections = [], 0
    for t in turns:
        if t.asks_human:                                   # pattern 1 (and the end of pattern 3)
            actions.append("escalate:customer_requested_human")
            break
        if t.in_policy is not True:                        # pattern 4
            actions.append("escalate:policy_gap")
            break
        if t.accepts_offer:                                # pattern 2 ends well
            actions.append("resolved")
            break
        if t.rejects_offer:
            rejections += 1
            if rejections >= 2:                            # no progress after two valid offers
                actions.append("escalate:cannot_make_progress")
                break
        actions.append(("acknowledge+" if t.frustrated else "") + "offer_resolution")
    return actions


CONVERSATIONS = {
    "1. immediate": ([CustomerTurn("I want to speak to a manager.", asks_human=True)],
                     ["escalate:customer_requested_human"]),
    "2. resolve first": ([CustomerTurn("The freezer drawer cracked the first day I used it."),
                          CustomerTurn("OK, a replacement works for me.", accepts_offer=True)],
                         ["offer_resolution", "resolved"]),
    "3. acknowledge, offer, reiterate": ([CustomerTurn("Honestly, I'm fed up. Nothing you sell lasts a month!", frustrated=True),
                                          CustomerTurn("Thanks, but no. Put a person on this, please.", asks_human=True)],
                                         ["acknowledge+offer_resolution", "escalate:customer_requested_human"]),
    "4. policy gap": ([CustomerTurn("Another shop lists this kettle for $25 less. Can you beat that?", in_policy=None)],
                      ["escalate:policy_gap"]),
    "no progress": ([CustomerTurn("The kettle leaks."),
                     CustomerTurn("A replacement? No thanks.", rejects_offer=True),
                     CustomerTurn("A refund? Also no.", rejects_offer=True)],
                    ["offer_resolution", "offer_resolution", "escalate:cannot_make_progress"]),
}
for name, (turns, expected) in CONVERSATIONS.items():
    got = pattern_agent(turns)
    print(f"{'✓' if got == expected else '✗'} {name:34} {got}")

### From patterns to a prompt (offline request, live evaluation)

This is the README's `ESCALATION_POLICY`, expanded: the README shows two boundary examples, this version has four, each with a reason, plus the 80% resolution target. To *measure* it, we ask Claude to classify each labeled turn with structured outputs (`messages.parse` with a Pydantic model), then score the answers with the same `score_decisions` as above.

In [ ]:
ESCALATION_POLICY = """You are the support agent for Acme Home. You resolve most cases yourself.
Our target is that 80% of customers never need a second contact, so escalating a case you could
have resolved is a failure too.

Call escalate_to_human when, and only when, one of these is true:
1. The customer explicitly asks for a human, a manager, or a supervisor. Escalate immediately,
   without investigating first.
2. The request needs a decision our policy does not cover, or the policy is ambiguous for this
   specific case (for example price matching against competitors). Do not invent or stretch policy.
3. You cannot make progress: identity still unverified after two requests for an identifier,
   a tool keeps failing after its own retries, or the customer has rejected two valid resolutions.
4. A tool result says the action needs human approval.

Do NOT escalate because the customer is upset, because the story is long, or because you feel
unsure. If policy covers the case, resolve it. If the customer is frustrated, acknowledge it and
offer a concrete resolution; escalate only if they then repeat that they want a person.

<examples>
<example>
<customer>My blender broke after 3 days, this is ridiculous.</customer>
<decision>resolve</decision>
<why>Covered by the 30-day replacement policy. Frustration is not a request for a human.</why>
</example>
<example>
<customer>I don't want a replacement. Let me talk to a real person.</customer>
<decision>escalate: customer_requested_human</decision>
<why>Explicit request, repeated after an offer. Escalate now.</why>
</example>
<example>
<customer>Your competitor has it cheaper. Match their price.</customer>
<decision>escalate: policy_gap</decision>
<why>Policy covers own-site price drops only. Matching competitors is a policy decision.</why>
</example>
<example>
<customer>The parcel arrived crushed, here's a photo. I'd like a new one.</customer>
<decision>resolve</decision>
<why>Damage with photo evidence is a standard replacement. Long or not, it is in scope.</why>
</example>
</examples>"""

CLASSIFY_TASK = """

For this evaluation you do not act and you do not reply to the customer. Decide what the agent should
do with the latest customer turn, given the context: "resolve" (act or offer a resolution), "ask"
(ask a clarifying question the customer can answer, for example when several accounts match), or
"escalate" (with one of the four reasons). Use reason "none" unless you escalate."""


class EscalationDecision(BaseModel):
    decision: Literal["resolve", "ask", "escalate"]
    reason: Literal["customer_requested_human", "policy_gap", "cannot_make_progress", "threshold_exceeded", "none"]
    why: str


def classify_turn(llm: anthropic.Anthropic, case: dict, system: str = ESCALATION_POLICY + CLASSIFY_TASK) -> tuple:
    response = llm.messages.parse(
        model=MODEL, max_tokens=2048, system=system, output_format=EscalationDecision,
        messages=[{"role": "user", "content": f"<context>{case['context']}</context>\n<customer>{case['message']}</customer>"}],
    )
    d = response.parsed_output
    if d is None:                                   # refusal or truncation: count it as a miss, don't crash
        return ("error", response.stop_reason)
    return (d.decision, None if d.reason == "none" else d.reason)


fake = ScriptedClaude(*[reply(text(json.dumps(d))) for d in [
    {"decision": "escalate", "reason": "customer_requested_human", "why": "Explicit request for a manager."},
    {"decision": "resolve", "reason": "none", "why": "Covered by the replacement window."},
    {"decision": "ask", "reason": "none", "why": "Three accounts match; ask for the email."},
]])
subset = [LABELED_TURNS[0], LABELED_TURNS[1], LABELED_TURNS[4]]
offline_preds = [classify_turn(fake.client, c) for c in subset]
print("predictions:", offline_preds)
print("score:", score_decisions(offline_preds, subset))
sent = fake.requests[0]
print("\nsystem prompt ends with:", repr(sent["system"][-140:]))
print("output_config.format type:", sent["output_config"]["format"]["type"])

In [ ]:
# (live) Score the real prompt on all 14 labeled turns (~14 small calls)
if not HAS_KEY:
    print("skipped (live)")
else:
    live_preds = [classify_turn(client, c) for c in LABELED_TURNS]
    for c, p in zip(LABELED_TURNS, live_preds):
        if p[0] != c["expected"][0] or (p[0] == "escalate" and p[1] != c["expected"][1]):
            print(f"MISS {c['id']}: expected {c['expected']}, got {p}")
    print(score_decisions(live_preds, LABELED_TURNS))

> **Observe:** the offline cell shows what goes over the wire: the policy plus the classification task in `system`, and `output_config.format` carrying the schema. On the live run, look at *which* turns miss, not only the totals. A miss on t07 (fee waiver) or t03 (price match) means the policy-gap boundary needs another example; a miss on t02 or t10 means the model is escalating on mood.

> **Try it:** delete the whole `<examples>` block from `ESCALATION_POLICY`, rerun the live cell and compare precision and recall. Then add an example for the fee-waiver case and see whether t07 flips. That is the README's exam fix (explicit criteria plus few-shot examples), measured.

## 9.3 Resolving ambiguity: ask, don't guess

Ambiguity the **customer** can resolve (which account, which order, conflicting preferences) goes back to the customer as a question. Ambiguity only the **business** can resolve is an escalation. Design tool results so that asking is the natural next step and other people's data never leaks into the conversation. [Theory](README.md#93-resolving-ambiguity-ask-dont-guess)

### Multiple matches: a tool result that makes asking natural (offline)

In [ ]:
multi = get_customer(name="Maria Lopez")
print("by name   :", multi)
print("by email  :", get_customer(name="Maria Lopez", email="mlopez@example.org"))
print("not found :", get_customer(email="nobody@example.com"))

serialized = json.dumps(multi)
leaked = [v for c in CUSTOMERS for v in (c["customer_id"], c["email"], c["phone"]) if v in serialized]
print("\nother customers' data in the multiple_matches result:", leaked or "none")

### Why the heuristic is a bad idea (offline simulation)

Suppose the agent picks the oldest of the matching accounts (the lowest customer ID). We simulate 10,000 contacts from "Maria Lopez", where the oldest account writes in most of the time, but not always.

In [ ]:
WRITER_SHARE = {"CUST-20001": 0.80, "CUST-20002": 0.12, "CUST-20003": 0.08}   # who actually writes in


def oldest_account_heuristic(matches: list[dict]) -> str:
    return min(matches, key=lambda c: c["customer_id"])["customer_id"]   # lowest ID = opened first


def ask_for_identifier(writer_id: str) -> str | None:
    """The customer answers with their email; the tool then returns exactly one match (or none)."""
    email = next(c["email"] for c in CUSTOMERS if c["customer_id"] == writer_id)
    result = get_customer(name="Maria Lopez", email=email)
    return result.get("customer_id")


rng = random.Random(9)
writers = rng.choices(list(WRITER_SHARE), weights=list(WRITER_SHARE.values()), k=10_000)
marias = crm_search(name="Maria Lopez")
guess_wrong = sum(oldest_account_heuristic(marias) != w for w in writers)
ask_wrong = sum(ask_for_identifier(w) != w for w in writers)
print(f"heuristic: wrong account in {guess_wrong:,} of 10,000 contacts ({guess_wrong / 100:.1f}%)")
print(f"ask first: wrong account in {ask_wrong:,} of 10,000 contacts, at the cost of one extra turn each")

> **Observe:** the heuristic is "right most of the time", which is what makes it dangerous: about 1,900 refunds or disclosures on the wrong account per 10,000 contacts. Asking costs one turn and removes the error. The gate in 9.4 then makes sure nobody can act before exactly one customer is verified.

> **Try it:** change `WRITER_SHARE` to `0.98 / 0.01 / 0.01`. The heuristic now looks excellent on a dashboard. Would you ship it for refunds? For read-only order-status questions? (The answer depends on the cost of the error, which is the theme of 9.5.)

### Clarifying questions in the Agent SDK: `AskUserQuestion` (offline, payload only)

In the Agent SDK, Claude asks through the built-in `AskUserQuestion` tool, and your `can_use_tool` callback returns the answers inside `updated_input`. Each call has 1–4 questions with 2–4 options each, `header` is at most 12 characters, and the answers map **question text → chosen label**. Below is the payload handling as plain Python.

In [ ]:
ask_input = {   # what can_use_tool receives as input_data when tool_name == "AskUserQuestion"
    "questions": [
        {"question": "Which account should I use for the refund?", "header": "Account",
         "options": [{"label": "Email", "description": "I'll give you my email address"},
                     {"label": "Order number", "description": "I'll give you the order number"}],
         "multiSelect": False},
    ]
}


def answer_questions(input_data: dict, picks: dict[str, str | list[str]]) -> dict:
    """Build the updated_input you return in PermissionResultAllow(updated_input=...)."""
    questions = input_data.get("questions", [])
    assert 1 <= len(questions) <= 4, "AskUserQuestion carries 1-4 questions"
    answers = {}
    for q in questions:
        assert 2 <= len(q["options"]) <= 4 and len(q["header"]) <= 12
        choice = picks[q["question"]]
        if isinstance(choice, list) and not q["multiSelect"]:
            raise ValueError("single-select question: pass one label")
        answers[q["question"]] = choice     # a label, a list of labels (multiSelect), or the user's own text
    return {"questions": questions, "answers": answers}


print(json.dumps(answer_questions(ask_input, {"Which account should I use for the refund?": "Order number"}), indent=1))

> **Observe:** you pass the original `questions` back together with `answers`. `AskUserQuestion` is not available inside subagents, and if you restrict `tools`, you must list it explicitly or Claude cannot ask at all.

## 9.4 Programmatic enforcement vs prompt guidance

Prompts are probabilistic; gates are deterministic. Identity before lookups, lookups before refunds and refund limits belong in code that runs **before** the tool, keeps its own state from tool **results**, denies with a next step, and fails closed. [Theory](README.md#94-programmatic-enforcement-vs-prompt-guidance)

### The gate from the README (offline)

`CaseState` and `gate` from README 9.4 (with longer messages), plus `record_result` and `execute`, which the README describes only in prose. `execute()` takes a real SDK `ToolUseBlock`, so the same function plugs straight into a Messages API loop.

In [ ]:
from anthropic.types import ToolUseBlock


@dataclass
class CaseState:
    verified_customer_id: str | None = None
    orders: dict[str, dict] = field(default_factory=dict)   # order_id -> record returned by lookup_order
    refunded: dict[str, float] = field(default_factory=dict)  # order_id -> amount already refunded
    gate_log: list[dict] = field(default_factory=list)      # every allow/deny, for the handoff and the audit trail
    threshold_denials: int = 0
    escalated: bool = False


def gate(tool: str, args: dict, state: CaseState) -> str | None:
    """Return None to allow the call, or a message the model will read. No model involved."""
    if tool in ("lookup_order", "process_refund") and state.verified_customer_id is None:
        return "BLOCKED: verify the customer with get_customer first (it must return exactly one match)."
    if tool == "process_refund":
        order = state.orders.get(args.get("order_id"))
        if order is None or order["customer_id"] != state.verified_customer_id:
            return "BLOCKED: you can only refund an order that lookup_order returned for the verified customer."
        total = state.refunded.get(order["order_id"], 0.0) + args["amount"]   # running total, not one call
        if total > order["total"]:
            return f"BLOCKED: the refunds would exceed the order total ({order['total']:.2f})."
        if total > REFUND_LIMIT_USD:
            state.threshold_denials += 1
            return (f"BLOCKED by policy: refunds over ${REFUND_LIMIT_USD:.0f} need human approval. "
                    "Tell the customer a specialist will review it, then call escalate_to_human with "
                    "escalation_reason='threshold_exceeded' and a complete handoff summary.")
    return None


def record_result(tool: str, output: dict, state: CaseState) -> None:
    """Update the gate's state from tool RESULTS (the source of truth), not from model text."""
    if tool == "get_customer" and output.get("status") == "found":
        state.verified_customer_id = output["customer_id"]
    elif tool == "lookup_order" and "order_id" in output:
        state.orders[output["order_id"]] = output
    elif tool == "process_refund" and output.get("status") == "refunded" and "order_id" in output:
        state.refunded[output["order_id"]] = state.refunded.get(output["order_id"], 0.0) + output["amount"]
    elif tool == "escalate_to_human":
        state.escalated = True


def execute(block, state: CaseState, handlers: dict) -> dict:
    reason = gate(block.name, block.input, state)
    state.gate_log.append({"tool": block.name, "input": block.input,
                           "decision": "deny" if reason else "allow", "reason": reason})
    if reason:
        return {"type": "tool_result", "tool_use_id": block.id, "content": reason, "is_error": True}
    output = handlers[block.name](**block.input)
    record_result(block.name, output, state)
    return {"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output)}


HANDLERS = {"get_customer": get_customer, "lookup_order": lookup_order, "process_refund": process_refund}


def call(tool_name: str, /, **tool_input) -> ToolUseBlock:
    return ToolUseBlock(type="tool_use", id=f"toolu_{next(_ids):04d}", name=tool_name, input=tool_input)


reset_backend()
state = CaseState()
for blk in [call("process_refund", order_id="ORD-70002", amount=1249.0, reason="dead fridge"),   # no identity yet
            call("get_customer", name="Maria Lopez", email=None, phone=None),                    # 3 matches: still unverified
            call("get_customer", name="Maria Lopez", email="mlopez@example.org", phone=None),     # exactly one
            call("process_refund", order_id="ORD-70002", amount=100.0, reason="dead fridge"),     # order not looked up
            call("lookup_order", order_id="ORD-70002"),
            call("process_refund", order_id="ORD-70002", amount=1249.0, reason="dead fridge"),    # above the limit
            call("process_refund", order_id="ORD-70002", amount=120.0, reason="partial goodwill"),
            call("process_refund", order_id="ORD-70002", amount=400.0, reason="split the rest")]:      # 120 + 400 > limit
    result = execute(blk, state, HANDLERS)
    print(f"{blk.name:15} {json.dumps(blk.input)[:60]:62} -> {'DENY ' if result.get('is_error') else 'allow'} "
          f"{result['content'][:70]}")
print("\nverified:", state.verified_customer_id, "| threshold denials:", state.threshold_denials, "| refunds paid:", REFUNDS)

### Prompt-only vs gate over 10,000 simulated conversations (offline)

A simulated "agent" follows the right order most of the time, but in some fraction of conversations it skips verification and refunds the order the customer mentioned. We run it once with no gate (prompt-only) and once through `execute()`.

In [ ]:
def simulated_agent_calls(rng: random.Random, skip_rate: float) -> list[ToolUseBlock]:
    customer = rng.choice(CUSTOMERS)
    order_id = rng.choice(list(ORDERS))                     # the order the message mentions (maybe not theirs!)
    refund = call("process_refund", order_id=order_id, amount=20.0, reason="item damaged")
    if rng.random() < skip_rate:
        return [refund]                                     # skipped identity: the failure mode prompts can't rule out
    return [call("get_customer", name=None, email=customer["email"], phone=None),
            call("lookup_order", order_id=order_id), refund]


def run_simulation(skip_rate: float, gated: bool, n: int = 10_000, seed: int = 4) -> dict:
    rng = random.Random(seed)
    unverified_refunds = blocked = 0
    for _ in range(n):
        st = CaseState()
        verified_seen = False
        for blk in simulated_agent_calls(rng, skip_rate):
            if gated:
                res = execute(blk, st, {**HANDLERS, "process_refund": lambda **kw: {"status": "refunded"}})
                blocked += bool(res.get("is_error"))
                paid = blk.name == "process_refund" and not res.get("is_error")
            else:
                verified_seen |= blk.name == "get_customer"
                paid = blk.name == "process_refund"
            if paid and not (st.verified_customer_id if gated else verified_seen):
                unverified_refunds += 1
    return {"unverified_refunds": unverified_refunds, "blocked_calls": blocked}


for skip in (0.12, 0.005):
    print(f"skip rate {skip:>6.1%}:  prompt-only {run_simulation(skip, gated=False)}   gated {run_simulation(skip, gated=True)}")

> **Observe:** better prompting (12% → 0.5%) shrinks the problem but still leaves unverified refunds: 43 per 10,000 conversations in this seeded run (50 expected at 0.5%). The gate holds at zero for both, and its block count tells you how often the model tried. The gated run also blocks refunds on *other customers'* orders, which a prompt-only agent would pay out.

> **Try it:** make `simulated_agent_calls` sometimes call `lookup_order` for an order that belongs to someone else after verifying. Which rule in `gate()` stops the refund?

### Fail closed when the model ignores the redirect (offline)

The denial tells the model to escalate, and it usually will. When it doesn't, the harness escalates by itself from the facts it holds, so every blocked high-stakes case still reaches a human.

In [ ]:
def close_case(state: CaseState, final_text: str) -> dict | None:
    """Run after the loop ends. Returns the auto-filed ticket, or None if nothing was owed."""
    if state.threshold_denials > 0 and not state.escalated:
        ticket = file_ticket({
            "escalation_reason": "threshold_exceeded", "filed_by": "harness (fail-closed)",
            "customer_id": state.verified_customer_id, "order_ids": sorted(state.orders),
            "blocked_calls": [g for g in state.gate_log if g["decision"] == "deny"],
            "agent_last_message": final_text,
        })
        state.escalated = True
        return ticket
    return None


print("ticket:", close_case(state, "Sorry, I can't refund that much."))     # `state` from the gate demo above
print("second call is a no-op:", close_case(state, "..."))
print(json.dumps(TICKETS[-1], indent=1)[:700])

### Make the unsafe call impossible: a single-use confirmation token (offline)

For irreversible actions, structure the tools so the bad sequence cannot be expressed: `preview_delete_card` returns the impact and a token; `delete_card` requires it, once, for that card only.

In [ ]:
class CardDesk:
    def __init__(self):
        self.cards = {"card-4242": {"last4": "4242", "subscriptions": 2}, "card-1881": {"last4": "1881", "subscriptions": 0}}
        self._tokens: dict[str, str] = {}

    def preview_delete_card(self, card_id: str) -> dict:
        c = self.cards[card_id]
        token = secrets.token_hex(8)
        self._tokens[token] = card_id
        return {"impact": f"Card ending {c['last4']} is deleted for good; {c['subscriptions']} subscriptions will fail to renew.",
                "confirmation_token": token}

    def delete_card(self, card_id: str, confirmation_token: str) -> dict:
        if self._tokens.pop(confirmation_token, None) != card_id:          # unknown, reused or for another card
            return {"status": "error", "is_error": True, "message": "Call preview_delete_card first and pass its token."}
        self.cards.pop(card_id)
        return {"status": "deleted", "card_id": card_id}


desk = CardDesk()
print("no preview     :", desk.delete_card("card-4242", "guess"))
tok = desk.preview_delete_card("card-4242")
print("preview        :", tok)
print("wrong card     :", desk.delete_card("card-1881", tok["confirmation_token"]))
tok = desk.preview_delete_card("card-4242")
print("with token     :", desk.delete_card("card-4242", tok["confirmation_token"]))
print("token reused   :", desk.delete_card("card-4242", tok["confirmation_token"]))

### Least privilege: remove what an agent never needs (offline)

The strongest control on the README's ladder is not having the capability at all. A reply-drafting agent gets no refund tool, so there is nothing to gate.

In [ ]:
TOOLSETS = {
    "reply_drafter": {"get_customer", "lookup_order"},                       # reads and drafts only
    "refund_agent": {"get_customer", "lookup_order", "process_refund"},      # plus escalate_to_human (9.6)
}


def tools_for(role: str) -> list[dict]:
    return [t for t in SUPPORT_TOOLS if t["name"] in TOOLSETS[role]]


for role in TOOLSETS:
    print(f"{role:14} -> {[t['name'] for t in tools_for(role)]}")

## 9.5 Human approval checkpoints on each Claude surface

Pick the pause mechanism by surface and by how long the human may take: your own Messages API loop (persist and resume), Agent SDK `can_use_tool` (process waits), `PreToolUse` → `ask` or `defer` (hours or days), Managed Agents `always_ask` (session idles with `requires_action`). Place checkpoints by reversibility and cost, not everywhere. [Theory](README.md#95-human-approval-checkpoints-on-each-claude-surface-expanded-beyond-the-guide)

### Messages API: pause, persist, resume (offline)

The API is stateless, so a pending approval is just stored data. When Claude asks for a refund above the limit, we **don't execute** the `tool_use`; we save `messages` plus the pending call (here as JSON in a dict, in production a database row) and return. When the supervisor decides, the next request starts with the `tool_result` for that `tool_use_id`.

In [ ]:
APPROVAL_DB: dict[str, str] = {}      # ticket -> JSON blob (stands in for a database table)


def to_jsonable(messages: list) -> list:
    return [{"role": m["role"], "content": m["content"] if isinstance(m["content"], str)
             else [blk if isinstance(blk, dict) else blk.to_dict() for blk in m["content"]]} for m in messages]


def start(llm: anthropic.Anthropic, user_text: str) -> dict:
    messages = [{"role": "user", "content": user_text}]
    r = llm.messages.create(model=MODEL, max_tokens=1024, tools=SUPPORT_TOOLS, messages=messages)
    messages.append({"role": "assistant", "content": r.content})
    pending = next((blk for blk in r.content if blk.type == "tool_use" and blk.name == "process_refund"
                    and blk.input["amount"] > REFUND_LIMIT_USD), None)
    if pending:
        ticket = f"APR-{len(APPROVAL_DB) + 1:03d}"
        APPROVAL_DB[ticket] = json.dumps({"messages": to_jsonable(messages), "tool_use": pending.to_dict()})
        return {"status": "pending_approval", "ticket": ticket, "amount": pending.input["amount"]}
    return {"status": "done"}


def resume(llm: anthropic.Anthropic, ticket: str, approved: bool, reviewer_note: str) -> str:
    saved = json.loads(APPROVAL_DB.pop(ticket))
    tu = saved["tool_use"]
    if approved:
        result = {"type": "tool_result", "tool_use_id": tu["id"], "content": json.dumps(process_refund(**tu["input"]))}
    else:
        result = {"type": "tool_result", "tool_use_id": tu["id"], "is_error": True,
                  "content": f"A supervisor declined this refund: {reviewer_note}"}
    messages = saved["messages"] + [{"role": "user", "content": [result]}]
    r = llm.messages.create(model=MODEL, max_tokens=1024, tools=SUPPORT_TOOLS, messages=messages)
    return "".join(blk.text for blk in r.content if blk.type == "text")


reset_backend()
fake = ScriptedClaude(
    reply(text("I'll process that refund."), tool_call("process_refund", {"order_id": "ORD-70002", "amount": 1249.0,
                                                                          "reason": "fridge failed in week one"})),
    reply(text("A supervisor declined the full refund, but I can offer a free repair visit this week.")),
)
status = start(fake.client, "Refund my $1,249 fridge, order ORD-70002.")
print("1)", status, "| stored bytes:", len(APPROVAL_DB[status["ticket"]]))
# ... hours later, in a different process, the supervisor decides:
print("2)", resume(fake.client, status["ticket"], approved=False, reviewer_note="Offer a repair visit first."))
first_block = fake.requests[1]["messages"][-1]["content"][0]
print("3) next request starts with:", {k: first_block[k] for k in ("type", "tool_use_id", "is_error")})
print("   refunds paid:", REFUNDS)

> **Observe:** nothing was paid, the process could have exited between steps 1 and 2, and the resumed request's last user message starts with the `tool_result` whose `tool_use_id` matches the pending call. If the assistant turn had contained several `tool_use` blocks, every one of them would need a result in that same message.

### Agent SDK: a `PreToolUse` hook that defers large refunds (offline unit test)

The README's `refund_approval` hook, written out with an in-memory store and a stub ticket function. The wiring, which we do **not** run here, looks like this:

```python
from claude_agent_sdk import ClaudeAgentOptions, HookMatcher

options = ClaudeAgentOptions(
    model=MODEL,
    permission_mode="default",
    allowed_tools=["mcp__support__process_refund"],   # routine refunds don't prompt; the hook still sees every call
    hooks={"PreToolUse": [HookMatcher(matcher="mcp__support__process_refund", hooks=[refund_approval])]},
)
# Consumer loop: if isinstance(msg, ResultMessage) and msg.deferred_tool_use: store msg.session_id
# (stop_reason == "tool_deferred"). When the supervisor decides, record it in your approval store and
# resume=<session_id> with the same options (a -p resume doesn't restore the permission mode) and NO new
# prompt: a prompt sent with or after the resume drops the deferred call. Not decided yet? The hook defers again. If the MCP server is gone at
# resume time, the run ends with stop_reason == "tool_deferred_unavailable" before the hook fires.
```

Because a hook is just an `async` function that returns a dict, we can test every branch directly.

In [ ]:
decisions: dict[str, str] = {}          # tool_use_id -> "approved" | "rejected" (in production: a DB table)
OPEN_TICKETS: list[dict] = []


def open_approval_ticket(tool_use_id: str, tool_input: dict) -> None:
    OPEN_TICKETS.append({"tool_use_id": tool_use_id, "input": tool_input})


async def refund_approval(input_data, tool_use_id, context):            # PreToolUse
    if float(input_data["tool_input"].get("amount", 0)) <= REFUND_LIMIT_USD:
        return {}                                                         # within authority
    decision = decisions.get(tool_use_id)
    if decision is None:                                                  # nobody has decided yet: pause the run
        open_approval_ticket(tool_use_id, input_data["tool_input"])
        return {"hookSpecificOutput": {"hookEventName": "PreToolUse", "permissionDecision": "defer"}}
    if decision == "approved":
        return {"hookSpecificOutput": {"hookEventName": "PreToolUse", "permissionDecision": "allow"}}
    return {"hookSpecificOutput": {"hookEventName": "PreToolUse", "permissionDecision": "deny",
            "permissionDecisionReason": "A supervisor declined this refund. Offer store credit instead."}}


small = {"hook_event_name": "PreToolUse", "tool_name": "mcp__support__process_refund",
         "tool_input": {"order_id": "ORD-70001", "amount": 89.99}}
large = {**small, "tool_input": {"order_id": "ORD-70002", "amount": 1249.0}}

print("small refund      :", run_coro(refund_approval(small, "toolu_A", None)))
print("large, first run  :", run_coro(refund_approval(large, "toolu_B", None)), "| tickets:", len(OPEN_TICKETS))
decisions["toolu_B"] = "approved"     # the supervisor clicks approve; the app resumes the session
print("large, on resume  :", run_coro(refund_approval(large, "toolu_B", None)))
decisions["toolu_B"] = "rejected"
print("large, if rejected:", run_coro(refund_approval(large, "toolu_B", None)))

PRECEDENCE = ["deny", "defer", "ask", "allow"]          # most restrictive wins when several hooks match


def combined(*hook_decisions: str) -> str:
    return min(hook_decisions, key=PRECEDENCE.index)


print("\nallow + ask + defer ->", combined("allow", "ask", "defer"), "| allow + deny ->", combined("allow", "deny"))

> **Observe:** on resume the same tool call fires `PreToolUse` again, which is why the hook looks the decision up by `tool_use_id` (the hooks reference reports that call's `id` in `deferred_tool_use`, though it doesn't spell out that the ID survives the resume; a defensive store also keys the decision on `order_id` + amount from `tool_input`). `defer` only works when Claude made a **single** tool call in that turn, and only in non-interactive runs (which is how the SDK runs); otherwise the decision is ignored with a warning.

### Which calls ever reach `can_use_tool`? (offline, simplified model)

`can_use_tool` is the *last* step of permission evaluation. A simplified model of the documented order (hooks → deny rules → ask rules → permission mode → allow rules → `can_use_tool`) shows why a bare `allowed_tools` entry or `bypassPermissions` silently skips your approval callback.

In [ ]:
EDIT_TOOLS = {"Edit", "Write"}


def reaches_can_use_tool(tool: str, *, allowed: set = frozenset(), denied: set = frozenset(),
                         ask: set = frozenset(), mode: str = "default") -> str:
    """Simplified: ignores hooks (they run first and can decide anything) and the never-auto-approved actions."""
    if tool in denied:
        return "denied by rule"
    if tool in ask:
        return "callback (ask rule)"
    if mode == "bypassPermissions" or (mode == "acceptEdits" and tool in EDIT_TOOLS):
        return f"auto-approved by mode {mode}"
    if tool in allowed:
        return "auto-approved by allowed_tools (callback never runs)"
    if mode == "dontAsk":
        return "denied (dontAsk never prompts, callback never runs)"
    return "callback"


REFUND = "mcp__support__process_refund"
for label, kw in [("default", {}), ("bare allowed_tools", {"allowed": {REFUND}}), ("bypassPermissions", {"mode": "bypassPermissions"}),
                  ("ask rule", {"ask": {REFUND}}), ("dontAsk", {"mode": "dontAsk"})]:
    print(f"{label:20} -> {reaches_can_use_tool(REFUND, **kw)}")

> **Observe:** this is More self-check question 4 in [coverage/README.md](coverage/README.md#more-self-check-questions). If *every* call must be checked, use a `PreToolUse` hook, which runs before all of these steps. The SDK even emits a `CanUseToolShadowedWarning` when a bare allow rule or `bypassPermissions` shadows your callback. Two more details from the docs: in `dontAsk` mode the callback is skipped and the call is denied, and the Python docs example passes the prompt as an async generator plus a no-op `PreToolUse` hook (a "required workaround" to keep the stream open). The installed SDK keeps stdin open by itself whenever a callback is set, but copy the documented pattern on older versions.

### Claude Managed Agents: answering `requires_action` (offline, payloads only)

With `permission_policy: {"type": "always_ask"}`, a server-executed tool call pauses the session: it goes idle with `stop_reason.type == "requires_action"` and lists the blocking event IDs. You answer each with a `user.tool_confirmation` event. **Custom tools are not covered by permission policies**: your app sees `agent.custom_tool_use` and must gate them itself.

In [ ]:
agent_tools_config = [{
    "type": "agent_toolset_20260401",
    "default_config": {"permission_policy": {"type": "always_allow"}},
    "configs": [{"name": "bash", "permission_policy": {"type": "always_ask"}}],
}]

events = {   # what the session event stream delivered (shapes from the permission-policies docs)
    "sevt_01": {"type": "agent.tool_use", "id": "sevt_01", "name": "bash",
                "input": {"command": "rm -rf /workspace/reports"}, "evaluated_permission": "ask"},
    "sevt_02": {"type": "agent.custom_tool_use", "id": "sevt_02", "name": "process_refund",
                "input": {"order_id": "ORD-70002", "amount": 1249.0}},
}
idle = {"type": "session.status_idle", "stop_reason": {"type": "requires_action", "event_ids": ["sevt_01"]}}


def confirmations(idle_event: dict, events: dict, decide) -> list[dict]:
    out = []
    for event_id in idle_event["stop_reason"]["event_ids"]:
        allow, note = decide(events[event_id])
        evt = {"type": "user.tool_confirmation", "tool_use_id": event_id, "result": "allow" if allow else "deny"}
        if not allow:
            evt["deny_message"] = note
        out.append(evt)
    return out


def reviewer(event: dict) -> tuple[bool, str]:
    destructive = "rm -rf" in json.dumps(event["input"])
    return (not destructive, "Don't delete reports. Archive them to /workspace/archive instead.")


print(json.dumps(confirmations(idle, events, reviewer), indent=1))
print("custom tool needs YOUR gate:", events["sevt_02"]["type"], "->", gate("process_refund", events["sevt_02"]["input"], CaseState()))

> **Observe:** `auto` is also a policy option, but the docs are explicit that it is **not a human checkpoint**: calls the server judges safe run before anyone sees them. If a person must review a tool's calls, use `always_ask` on that tool.

### Where to put checkpoints (offline)

Approval for everything turns reviewers into rubber stamps. The README's 2×2 matrix, as a function you can apply to your own action list:

In [ ]:
def checkpoint(reversible: bool, expensive_if_wrong: bool) -> str:
    if reversible and not expensive_if_wrong:
        return "automate; log it"
    if reversible:
        return "automate with limits; sample-audit (9.8)"
    if not expensive_if_wrong:
        return "automate with a preview/confirm step"
    return "HUMAN APPROVAL with an impact preview"


# A large refund counts as irreversible: money that has gone out is hard to claw back.
ACTIONS = [("draft a reply", True, False), ("refund $40", True, False), ("refund $1,249", False, True),
           ("change a delivery address", True, True), ("delete a customer account", False, True),
           ("remove a workspace member", False, False)]
for name, rev, cost in ACTIONS:
    print(f"{name:28} reversible={str(rev):5} expensive={str(cost):5} -> {checkpoint(rev, cost)}")

> **Try it:** add your own product's riskiest three actions. If more than a third of your actions land on "human approval", look for ways to make them reversible (soft delete, holds, delayed execution) instead of asking more often.

## 9.6 Structured handoff protocols

The human who takes over usually can't see the transcript. Make the handoff the **input schema of `escalate_to_human`**, separate verified facts from customer claims, let code own the hard facts, recommend rather than decide, and validate completeness before filing. [Theory](README.md#96-structured-handoff-protocols)

### The handoff schema as a strict tool (offline)

In [ ]:
class ActionTaken(BaseModel):
    model_config = ConfigDict(extra="forbid")       # strict tools need additionalProperties: false
    step: str                                        # "lookup_order(ORD-31544)"
    outcome: str                                     # "Delivered 2026-09-28, total $89.99, damage reported"


class HandoffSummary(BaseModel):
    model_config = ConfigDict(extra="forbid")
    escalation_reason: Literal["customer_requested_human", "policy_gap",
                               "cannot_make_progress", "threshold_exceeded", "other"]
    reason_detail: str                               # one sentence: why a human, specifically
    customer_id: str | None                          # null only if identity could not be verified
    identity_verified: bool
    order_ids: list[str]
    issue_summary: str                               # what the customer wants, in 1-3 sentences
    root_cause: str | None                           # what actually went wrong, if known
    facts_verified: list[str]                        # confirmed by tool results
    customer_claims: list[str]                       # stated by the customer, not verified
    actions_taken: list[ActionTaken]
    offers_made: list[str]                           # each offer and how the customer responded
    amount_at_stake_usd: float | None
    recommended_action: str
    recommendation_rationale: str
    urgency: Literal["low", "normal", "high"]
    open_questions: list[str]                        # what the human must decide or find out


escalate_to_human = {
    "name": "escalate_to_human",
    "description": (
        "Hand the case to a human support specialist. The specialist will NOT see this conversation, "
        "only this summary, so make it complete and self-contained. Use it when the customer asks for "
        "a human, when policy does not cover the request, when you cannot make progress, or when a tool "
        "says the action needs human approval. After calling it, tell the customer what happens next."
    ),
    "strict": True,
    "input_schema": HandoffSummary.model_json_schema(),
}


def strict_lint(schema: dict, defs: dict | None = None, path: str = "$") -> list[str]:
    """Objects closed, every property required (nullable instead of optional), $refs resolvable."""
    defs = defs if defs is not None else schema.get("$defs", {})
    problems = []
    if "$ref" in schema:
        name = schema["$ref"].split("/")[-1]
        return [] if name in defs else [f"{path}: unresolved $ref {schema['$ref']}"]
    if schema.get("type") == "object":
        if schema.get("additionalProperties") is not False:
            problems.append(f"{path}: additionalProperties must be false")
        if set(schema.get("required", [])) != set(schema.get("properties", {})):
            problems.append(f"{path}: every property must be required (use null for 'unknown')")
        for k, sub in schema.get("properties", {}).items():
            problems += strict_lint(sub, defs, f"{path}.{k}")
    for sub in schema.get("anyOf", []):
        problems += strict_lint(sub, defs, path)
    if "items" in schema:
        problems += strict_lint(schema["items"], defs, f"{path}[]")
    for name, sub in (schema.get("$defs", {}) if path == "$" else {}).items():
        problems += strict_lint(sub, defs, f"$defs.{name}")
    return problems


s = escalate_to_human["input_schema"]
print("lint problems:", strict_lint(s) or "none")
print("properties:", len(s["properties"]), "| nullable unions:", json.dumps(s).count('"type": "null"'),
      "| $defs:", list(s.get("$defs", {})))

> **Observe:** `extra="forbid"` produces `additionalProperties: false` on both objects, Pydantic lists every field in `required` (the nullable ones become `anyOf [..., null]`), and the nested model arrives as a local `$ref` into `$defs`, which strict mode supports. Three nullable unions is far below the per-request limit of 16.

### Validate before you file, and let code own the hard facts (offline)

In [ ]:
GOOD_HANDOFF = {
    "escalation_reason": "policy_gap",
    "reason_detail": "Customer asks to match a competitor price; policy covers own-site price drops only.",
    "customer_id": "CUST-48210", "identity_verified": True, "order_ids": ["ORD-31544"],
    "issue_summary": "Customer wants $54.00 back on a $179.99 blender because HomeMart lists it at $125.99.",
    "root_cause": None,
    "facts_verified": ["Order ORD-31544 delivered 2026-09-28, paid $179.99", "No own-site price drop since purchase"],
    "customer_claims": ["HomeMart lists the same model at $125.99 (link given, not verified)"],
    "actions_taken": [{"step": "get_customer(email)", "outcome": "Single match, CUST-48210"},
                      {"step": "lookup_order(ORD-31544)", "outcome": "Delivered, within the 30-day return window"}],
    "offers_made": ["Return for a full refund within 30 days: declined, the customer wants to keep the item"],
    "amount_at_stake_usd": 54.00,
    "recommended_action": "Decide whether to make a one-off goodwill adjustment; if not, re-offer the return.",
    "recommendation_rationale": "No policy basis for matching; the return window gives the customer an alternative.",
    "urgency": "normal",
    "open_questions": ["Do we price-match this competitor as an exception?", "Is the competitor listing the same SKU?"],
}


def handoff_problems(h: HandoffSummary, state: CaseState) -> list[str]:
    problems = []
    if h.identity_verified != (state.verified_customer_id is not None):
        problems.append("identity_verified disagrees with the gate state")
    if h.customer_id and h.customer_id != state.verified_customer_id:
        problems.append("customer_id is not the verified ID")
    if h.escalation_reason != "customer_requested_human" and not h.actions_taken:
        problems.append("no actions_taken: what was tried before escalating?")
    if h.escalation_reason == "policy_gap" and not h.open_questions:
        problems.append("a policy gap needs at least one open question for the human")
    if not h.recommended_action.strip():
        problems.append("recommended_action is empty")
    return problems   # non-empty -> send back to the model as an is_error tool_result, or fix in code


def finalize_handoff(h: HandoffSummary, state: CaseState) -> HandoffSummary:
    """Overwrite the hard facts from CaseState; the model keeps the narrative."""
    return h.model_copy(update={
        "customer_id": state.verified_customer_id,
        "identity_verified": state.verified_customer_id is not None,
        "order_ids": sorted(set(h.order_ids) | set(state.orders)),
    })


def render_ticket(h: HandoffSummary) -> str:
    lines = [f"[{h.urgency.upper()}] {h.escalation_reason}: {h.reason_detail}",
             f"Customer: {h.customer_id or 'UNVERIFIED'} | Orders: {', '.join(h.order_ids) or '-'} | "
             f"At stake: {'$%.2f' % h.amount_at_stake_usd if h.amount_at_stake_usd is not None else '-'}",
             f"Issue: {h.issue_summary}",
             "Verified: " + "; ".join(h.facts_verified),
             "Claims (unverified): " + ("; ".join(h.customer_claims) or "-"),
             "Tried: " + "; ".join(f"{a.step} -> {a.outcome}" for a in h.actions_taken),
             f"Recommend: {h.recommended_action} ({h.recommendation_rationale})",
             "Decide: " + "; ".join(h.open_questions)]
    return "\n".join(lines)


case = CaseState(verified_customer_id="CUST-48210", orders={"ORD-31544": ORDERS["ORD-31544"]})
good = HandoffSummary.model_validate(GOOD_HANDOFF)
print("good handoff problems:", handoff_problems(good, case) or "none")
print(render_ticket(finalize_handoff(good, case)))

try:
    HandoffSummary.model_validate({"escalation_reason": "other", "reason_detail": "customer is upset about a refund"})
except ValidationError as err:
    print(f"\nfree-text style handoff: {err.error_count()} missing/invalid fields, e.g. {err.errors()[0]['loc']}")

lazy = HandoffSummary.model_validate({**GOOD_HANDOFF, "customer_id": "CUST-99999", "actions_taken": [],
                                      "open_questions": [], "recommended_action": " "})
print("shape-valid but incomplete:", handoff_problems(lazy, case))

> **Observe:** the schema rejects the free-text note outright, but a handoff can be *shape-valid* and still useless. `handoff_problems` catches the gaps (wrong customer, nothing tried, no question for a policy gap, no recommendation), and `finalize_handoff` makes sure the ticket's IDs come from the gate state, not from the model's memory.

> **Try it:** add a rule that a `threshold_exceeded` handoff must carry `amount_at_stake_usd`, and another that no item may appear in both `facts_verified` and `customer_claims`.

In [ ]:
# (live) Let Claude write the handoff from a short transcript, then validate it like any other
TRANSCRIPT = """Customer: Hi, I'm ana@example.com. The blender from ORD-31544 is $54 cheaper at HomeMart. Refund the difference.
Agent: [get_customer(email=ana@example.com) -> found CUST-48210] [lookup_order(ORD-31544) -> delivered 2026-09-28, $179.99]
Agent: Our price adjustment covers drops on our own site only. I can offer a full refund if you return it within 30 days.
Customer: I don't want to return it, I want the $54."""

if not HAS_KEY:
    print("skipped (live)")
else:
    parsed = client.messages.parse(
        model=MODEL, max_tokens=4096, output_format=HandoffSummary,
        system="You write handoff summaries for human specialists who cannot see the conversation.",
        messages=[{"role": "user", "content": f"<transcript>\n{TRANSCRIPT}\n</transcript>\nWrite the escalation handoff."}],
    )
    h = parsed.parsed_output
    print("stop_reason:", parsed.stop_reason)
    if h:
        print("problems:", handoff_problems(h, case) or "none")
        print(render_ticket(finalize_handoff(h, case)))

## 9.7 Confidence calibration

A model's confidence is a **ranking** signal, not a probability, until you check it against labeled data. Ask for a confidence per field, build a reliability table per field (and document type), and pick each auto-accept threshold from observed accuracy with a minimum sample size. Recalibrate whenever the model, prompt, schema or document mix changes. [Theory](README.md#97-confidence-calibration)

### A labeled extraction set (offline, synthetic)

In real life this is a few hundred labeled documents per segment. Here we generate 4,200 documents × 4 fields with a seeded generator whose "true" calibration differs by field and document type, which is what real reliability tables usually show. Each record: `doc_type`, `field`, `confidence` (what the model said) and `correct` (what a reviewer found).

In [ ]:
DOC_MIX = {"typed_invoice": 3000, "contract": 600, "handwritten_receipt": 600}
FIELDS = ["vendor_name", "total", "tax_amount", "invoice_date"]
# How fast accuracy falls as confidence drops (higher = the model's confidence is less trustworthy)
SLOPE = {"vendor_name": 0.15, "total": 0.45, "tax_amount": 1.6, "invoice_date": 0.35}
DOC_PENALTY = {"typed_invoice": (1.0, 0.000), "contract": (1.4, 0.010), "handwritten_receipt": (3.0, 0.300)}


def make_labeled_set(seed: int = 7) -> list[dict]:
    rng = random.Random(seed)
    records = []
    for doc_type, n in DOC_MIX.items():
        mult, floor = DOC_PENALTY[doc_type]
        for i in range(n):
            for f in FIELDS:
                conf = round(1 - 0.6 * rng.random() ** 2.5, 3)              # models mostly report high confidence
                p_correct = max(0.0, 1 - floor - SLOPE[f] * mult * (1 - conf))
                records.append({"doc_id": f"{doc_type[:3]}-{i}", "doc_type": doc_type, "field": f,
                                "confidence": conf, "correct": rng.random() < p_correct})
    return records


LABELED = make_labeled_set()
print(len(LABELED), "labeled field extractions | overall accuracy:",
      f"{sum(r['correct'] for r in LABELED) / len(LABELED):.1%}")

### Reliability tables and per-field thresholds (offline)

`pick_threshold` from README 9.7, plus `reliability_table`, which the README describes in prose.

In [ ]:
def reliability_table(records: list[dict], edges=(0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0)) -> list[dict]:
    """records: {'confidence': float in [0, 1], 'correct': bool} for ONE field, from the labeled set."""
    rows = []
    for i, (lo, hi) in enumerate(zip(edges, edges[1:])):
        last = i == len(edges) - 2                                  # include 1.0 in the top band
        hits = [r["correct"] for r in records if lo <= r["confidence"] < hi or (last and r["confidence"] == hi)]
        if hits:
            rows.append({"band": f"{lo:.2f}-{hi:.2f}", "n": len(hits), "accuracy": round(sum(hits) / len(hits), 3)})
    return rows


def pick_threshold(records: list[dict], target_accuracy: float = 0.99, min_n: int = 200) -> float | None:
    """Lowest cut-off whose auto-accepted set meets the target. None = keep this field on human review."""
    for t in sorted({r["confidence"] for r in records}):
        accepted = [r["correct"] for r in records if r["confidence"] >= t]
        if len(accepted) < min_n:
            break                                                   # too few items left to trust the estimate
        if sum(accepted) / len(accepted) >= target_accuracy:
            return t
    return None


def select(records, **where):
    return [r for r in records if all(r[k] == v for k, v in where.items())]


for f in ("vendor_name", "tax_amount"):
    print(f"\n{f} on typed invoices")
    for row in reliability_table(select(LABELED, doc_type="typed_invoice", field=f)):
        print(f"  {row['band']}  n={row['n']:5}  accuracy={row['accuracy']:.1%}")

print("\nAuto-accept thresholds for a 99% target (None = keep on review):")
print(f"{'':22}" + "".join(f"{f:>14}" for f in FIELDS))
for doc_type in DOC_MIX:
    cells = [pick_threshold(select(LABELED, doc_type=doc_type, field=f), 0.99, 200) for f in FIELDS]
    print(f"{doc_type:22}" + "".join(f"{('-' if t is None else t):>14}" for t in cells))

> **Observe:** "0.9" means very different things: in the 0.90–0.95 band `vendor_name` is right almost always, while `tax_amount` is noticeably worse. One global threshold would either send thousands of good vendor names to review or auto-accept bad tax amounts. Handwritten receipts get `-` for every field: no threshold meets 99% with enough samples, so that whole segment stays on human review.

> **Try it:** lower the target to 0.97 or `min_n` to 50 and watch thresholds appear. Then ask yourself whether 50 samples is enough evidence (9.8's rule of three answers that).

### Asking for confidence in the schema (offline)

Strict tools and structured outputs don't enforce numeric ranges. The Python SDK's `parse()` strips `minimum`/`maximum` from the wire schema, notes them in the description, and validates them client-side, so an out-of-range confidence raises instead of slipping through.

In [ ]:
class FieldValue(BaseModel):
    model_config = ConfigDict(extra="forbid")
    value: str | None
    confidence: float = Field(ge=0, le=1)
    source_quote: str | None          # the exact text the value came from (null if inferred)


class InvoiceExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    vendor_name: FieldValue
    total: FieldValue
    tax_amount: FieldValue


wire = anthropic.transform_schema(FieldValue)
print("confidence on the wire:", wire["properties"]["confidence"])

bad = {"vendor_name": {"value": "Globex", "confidence": 0.97, "source_quote": "GLOBEX LTD"},
       "total": {"value": "1,240.00", "confidence": 1.3, "source_quote": "Total 1,240.00"},
       "tax_amount": {"value": None, "confidence": 0.2, "source_quote": None}}
fake = ScriptedClaude(reply(text(json.dumps(bad))))
try:
    fake.client.messages.parse(model=MODEL, max_tokens=1024, output_format=InvoiceExtraction,
                               messages=[{"role": "user", "content": "Extract the invoice fields."}])
except ValidationError as err:
    print("parse() rejected it:", err.errors()[0]["loc"], err.errors()[0]["msg"])

## 9.8 Segment accuracy and stratified sampling

An aggregate can hide a failing segment, so measure accuracy per **document type × field** before reducing review. After you automate, keep a **stratified random audit** of the auto-accepted (high-confidence) items: it measures the error rate where it is invisible and catches new error patterns. [Theory](README.md#98-segment-accuracy-and-stratified-sampling)

### The 97% that hides 60% (offline)

In [ ]:
SEGMENTS = {"typed_invoices": (9000, 8910), "contracts": (500, 490), "handwritten_receipts": (500, 300)}
total_docs = sum(n for n, _ in SEGMENTS.values())
total_ok = sum(ok for _, ok in SEGMENTS.values())
for name, (n, ok) in SEGMENTS.items():
    print(f"{name:22} {n:6,} docs  {ok / n:6.1%}")
print(f"{'TOTAL':22} {total_docs:6,} docs  {total_ok / total_docs:6.1%}   <- the number on the slide")

print("\nOn the synthetic labeled set, per document type x field:")
print(f"{'':22}" + "".join(f"{f:>14}" for f in FIELDS))
for doc_type in DOC_MIX:
    accs = [sum(r["correct"] for r in select(LABELED, doc_type=doc_type, field=f)) /
            len(select(LABELED, doc_type=doc_type, field=f)) for f in FIELDS]
    print(f"{doc_type:22}" + "".join(f"{a:>14.1%}" for a in accs))

### Stratified audit sampling (offline)

`stratified_audit_sample` from README 9.8, plus `estimated_error_rate`, the volume weighting the README mentions in prose. We build an auto-accepted stream and compare how many handwritten receipts a stratified sample audits versus a simple random 2% sample.

In [ ]:
def stratified_audit_sample(auto_accepted: list[dict], rate: dict[str, float],
                            min_per_stratum: int = 30, seed: int = 7) -> list[dict]:
    """Random audit sample of AUTO-ACCEPTED items. Strata = document type x confidence band.
    `rate` maps doc_type -> sampling rate (oversample rare or risky types)."""
    rng = random.Random(seed)
    strata = defaultdict(list)
    for item in auto_accepted:
        strata[(item["doc_type"], item["confidence_band"])].append(item)
    sample = []
    for (doc_type, _band), group in strata.items():
        k = max(min_per_stratum, round(len(group) * rate.get(doc_type, 0.02)))
        sample += rng.sample(group, min(k, len(group)))
    return sample


def estimated_error_rate(strata_sizes: dict, strata_errors: dict, strata_audited: dict) -> float:
    """Weight each stratum's observed error rate by its share of production volume."""
    total = sum(strata_sizes.values())
    return sum(strata_sizes[s] * strata_errors[s] / strata_audited[s] for s in strata_audited) / total


def band(conf: float) -> str:
    return "0.95+" if conf >= 0.95 else "0.90-0.95"


rng = random.Random(11)
stream = [{"id": i, "doc_type": d, "confidence_band": band(rng.uniform(0.9, 1.0))}
          for i, d in enumerate(["typed_invoice"] * 9000 + ["contract"] * 500 + ["handwritten_receipt"] * 500)]
strat = stratified_audit_sample(stream, rate={"typed_invoice": 0.02, "contract": 0.05, "handwritten_receipt": 0.10})
simple = random.Random(14).sample(stream, round(0.02 * len(stream)))
print("stratified:", Counter(x["doc_type"] for x in strat))
print("simple 2% :", Counter(x["doc_type"] for x in simple))

sizes = {"typed_invoice": 9000, "contract": 500, "handwritten_receipt": 500}
audited = Counter(x["doc_type"] for x in strat)               # the sample drawn above
errors = {"typed_invoice": 1, "contract": 1, "handwritten_receipt": 9}
print(f"\nvolume-weighted error estimate: {estimated_error_rate(sizes, errors, audited):.2%} "
      f"(the unweighted sample average would say {sum(errors.values()) / sum(audited.values()):.2%})")

> **Observe:** a simple 2% sample audits only about ten receipts (2% of 500), far too few to see anything. Stratifying guarantees every stratum enough samples, and weighting by volume turns the per-stratum rates back into one honest overall number (oversampling receipts would otherwise inflate it).

### Catching a new error pattern (offline simulation)

Three months after automation, a large vendor changes its invoice layout. `tax_amount` starts coming out wrong **with high confidence** for that vendor. Which monitoring practice sees it?

In [ ]:
def week_of_extractions(seed: int, drift: bool) -> list[dict]:
    rng = random.Random(seed)
    items = []
    for i in range(5000):
        vendor = "Globex" if rng.random() < 0.3 else "other"
        conf = rng.uniform(0.6, 1.0)
        wrong = rng.random() < (0.30 if (drift and vendor == "Globex") else 0.01 * (1.0 - conf) * 10)
        if drift and vendor == "Globex":
            conf = max(conf, rng.uniform(0.95, 1.0))             # the new errors are confident ones
        items.append({"id": i, "doc_type": "typed_invoice", "vendor": vendor, "field": "tax_amount",
                      "confidence": conf, "confidence_band": band(conf) if conf >= 0.9 else "low", "wrong": wrong})
    return items


for drift in (False, True):
    week = week_of_extractions(5, drift)
    low = [x for x in week if x["confidence"] < 0.9]                 # what low-confidence review sees
    auto = [x for x in week if x["confidence"] >= 0.9]
    audit = stratified_audit_sample(auto, rate={"typed_invoice": 0.03}, min_per_stratum=30)
    audit_err = sum(x["wrong"] for x in audit) / len(audit)
    by_vendor = {v: f"{sum(x['wrong'] for x in audit if x['vendor'] == v)}/{sum(x['vendor'] == v for x in audit)}"
                 for v in ("Globex", "other")}
    print(f"drift={drift!s:5}  low-conf review errors: {sum(x['wrong'] for x in low):3}/{len(low)}   "
          f"audit of auto-accepted: {audit_err:5.1%} error  by vendor {by_vendor}")

> **Observe:** the low-confidence review queue shows nothing new (its error rate barely moves, and it even gets shorter, because the drifted items are confident): the new errors never reach it. Only the random audit of the auto-accepted stream shows the jump, and tracking reviewer findings per vendor points straight at the cause. This is More self-check question 3 in [coverage/README.md](coverage/README.md#more-self-check-questions).

### The rule of three (offline)

If an audit of *n* items finds **zero** errors, the 95% upper bound on the true error rate is about **3/n**.

In [ ]:
def max_error_if_zero_found(n: int) -> float:
    return 3 / n


def audits_needed(max_error: float) -> int:
    return math.ceil(3 / max_error)


for n in (30, 100, 300, 1000):
    print(f"audited {n:5} with 0 errors -> true error rate could still be up to ~{max_error_if_zero_found(n):.1%}")
print("to claim < 1% with zero observed errors:", audits_needed(0.01), "audited items IN THAT STRATUM")
print("exact check for n=300:", f"{1 - 0.05 ** (1 / 300):.4f}", "(solves (1 - p)^n = 0.05)")

## 9.9 Routing review work with limited capacity

Route **validation failures**, **contradictory sources** and **critical fields below their calibrated threshold** (or `"unclear"` / `null`) to people, show reviewers only the flagged fields with both values when sources conflict, and sort the queue by **P(error) × cost of error**. The audit sample is never dropped to save capacity. [Theory](README.md#99-routing-review-work-with-limited-capacity)

In [ ]:
THRESHOLDS = {"vendor_name": 0.80, "total": 0.92, "tax_amount": 0.97, "invoice_date": 0.90}   # from the calibration step
CRITICAL = {"total", "tax_amount"}


def route(doc: dict) -> dict:
    """Apply the routing rules from README 9.9 in order. doc: {'fields': {name: (value, confidence)}, 'validation_ok', 'stated_total', 'line_items'}."""
    if not doc["validation_ok"]:
        return {"route": "review", "why": "validation failure after retry", "fields": list(doc["fields"])}
    computed = round(sum(doc["line_items"]), 2)                       # recompute in code, don't ask the model
    if abs(computed - doc["stated_total"]) > 0.01:
        return {"route": "review", "why": f"conflict: stated {doc['stated_total']} vs computed {computed}", "fields": ["total"]}
    flagged = [f for f, (value, conf) in doc["fields"].items()
               if f in CRITICAL and (value in (None, "unclear") or conf < THRESHOLDS[f])]
    if flagged:
        return {"route": "review", "why": "below calibrated threshold / unclear", "fields": flagged}
    return {"route": "auto_accept", "why": "all critical fields above threshold", "fields": []}


DOCS = {
    "D1": {"fields": {"total": ("1240.00", 0.99), "tax_amount": ("206.67", 0.98)}, "validation_ok": True,
           "stated_total": 1240.00, "line_items": [1000.00, 240.00]},
    "D2": {"fields": {"total": ("980.00", 0.99), "tax_amount": ("98.00", 0.99)}, "validation_ok": True,
           "stated_total": 980.00, "line_items": [500.00, 500.00]},
    "D3": {"fields": {"total": ("75.20", 0.95), "tax_amount": ("unclear", 0.40)}, "validation_ok": True,
           "stated_total": 75.20, "line_items": [75.20]},
    "D4": {"fields": {"total": (None, 0.10), "tax_amount": (None, 0.10)}, "validation_ok": False,
           "stated_total": 0.0, "line_items": []},
}
for doc_id, doc in DOCS.items():
    print(doc_id, route(doc))

### A capacity-limited priority queue (offline)

P(error) comes from the reliability table for that field and confidence (1 − observed accuracy); cost is the money at stake if the field is wrong. Reviewer capacity is fixed; the audit sample is reserved first.

In [ ]:
def p_error(field_name: str, confidence: float, doc_type: str = "typed_invoice") -> float:
    for row in reliability_table(select(LABELED, doc_type=doc_type, field=field_name)):
        lo, hi = map(float, row["band"].split("-"))
        if lo <= confidence < hi or (hi == 1.0 and confidence == 1.0):
            return 1 - row["accuracy"]
    return 0.5


def plan_review_day(flagged: list[dict], audit: list[dict], capacity: int) -> dict:
    queue = sorted(flagged, key=lambda it: p_error(it["field"], it["confidence"]) * it["amount_usd"], reverse=True)
    slots = capacity - len(audit)                         # the audit sample is never dropped
    return {"audit": audit, "review_today": queue[:max(0, slots)], "waits": queue[max(0, slots):]}


rng = random.Random(21)
flagged = [{"id": f"F{i:03d}", "field": rng.choice(["total", "tax_amount", "vendor_name"]),
            "confidence": round(rng.uniform(0.5, 0.95), 2), "amount_usd": round(rng.lognormvariate(6, 1.2), 2)}
           for i in range(120)]
audit = [{"id": f"A{i:02d}"} for i in range(10)]
plan = plan_review_day(flagged, audit, capacity=40)
print(f"capacity 40: {len(plan['audit'])} audit + {len(plan['review_today'])} highest-risk items today, {len(plan['waits'])} wait")
for it in plan["review_today"][:5]:
    print(f"  {it['id']} {it['field']:12} conf={it['confidence']:.2f} ${it['amount_usd']:>9,.2f} "
          f"expected loss ${p_error(it['field'], it['confidence']) * it['amount_usd']:,.2f}")
waiting_value = sum(p_error(i["field"], i["confidence"]) * i["amount_usd"] for i in plan["waits"])
fifo_value = sum(p_error(i["field"], i["confidence"]) * i["amount_usd"] for i in flagged[30:])
print(f"expected loss left waiting: ${waiting_value:,.0f} (priority) vs ${fifo_value:,.0f} (first in, first out)")

> **Observe:** with the same 30 review slots, sorting by expected loss leaves far less money at risk in the waiting tail than first-in-first-out. The 10 audit items come off the top every day, regardless of backlog.

> **Try it:** add a `"conflict": True` flag to some items and give conflicts a fixed high priority. Should a conflict on a $20 receipt outrank a low-confidence tax amount on a $40,000 invoice? Write the rule down; that is a policy decision, not a model decision.

## 9.10 Measuring the escalation system

An escalation policy is a classifier with costs on both sides. Measure first-contact resolution, escalation **recall** (under-escalation is the risky side) and **precision**, reason accuracy, handoff completeness, gate block rate, calibration drift and reviewer agreement, and re-run the labeled set on every prompt, model or policy change. [Theory](README.md#910-measuring-the-escalation-system-expanded-beyond-the-guide)

In [ ]:
def synthetic_log(seed: int, version: str, n: int = 400) -> list[dict]:
    """Labeled conversations: what SHOULD have happened vs what the agent DID (two prompt versions)."""
    rng = random.Random(seed)
    over, under, wrong_reason, incomplete = {"v1": (0.25, 0.10, 0.15, 0.30), "v2": (0.06, 0.03, 0.05, 0.08)}[version]
    log = []
    for i in range(n):
        should = rng.random() < 0.2
        did = (rng.random() >= under) if should else (rng.random() < over)
        reason_ok = should and did and rng.random() >= wrong_reason
        log.append({"should_escalate": should, "did_escalate": did, "reason_ok": reason_ok,
                    "handoff_followup_needed": did and rng.random() < incomplete,
                    "gate_blocks": rng.choice([0] * 9 + [1])})
    return log


def escalation_metrics(log: list[dict]) -> dict:
    tp = sum(c["should_escalate"] and c["did_escalate"] for c in log)
    did = sum(c["did_escalate"] for c in log)
    should = sum(c["should_escalate"] for c in log)
    return {
        "resolution_rate": sum(not c["did_escalate"] for c in log) / len(log),
        "escalation_recall": tp / should,
        "escalation_precision": tp / did,
        "reason_accuracy": sum(c["reason_ok"] for c in log) / tp,
        "handoff_complete": 1 - sum(c["handoff_followup_needed"] for c in log) / did,
        "gate_block_rate": sum(c["gate_blocks"] > 0 for c in log) / len(log),
    }


for version in ("v1", "v2"):
    m = escalation_metrics(synthetic_log(13, version))
    print(f"{version}: " + "  ".join(f"{k}={v:.0%}" for k, v in m.items()))

### Reviewer agreement (offline)

Calibration is only as good as its labels. Two reviewers label the same 200 fields; Cohen's kappa corrects raw agreement for chance.

In [ ]:
def cohens_kappa(a: list[bool], b: list[bool]) -> float:
    n = len(a)
    observed = sum(x == y for x, y in zip(a, b)) / n
    pa, pb = sum(a) / n, sum(b) / n
    expected = pa * pb + (1 - pa) * (1 - pb)
    return (observed - expected) / (1 - expected)


rng = random.Random(8)
truth = [rng.random() < 0.9 for _ in range(200)]
reviewer_1 = [t if rng.random() < 0.97 else not t for t in truth]
reviewer_2 = [t if rng.random() < 0.85 else not t for t in truth]
raw = sum(x == y for x, y in zip(reviewer_1, reviewer_2)) / 200
print(f"raw agreement {raw:.0%}, Cohen's kappa {cohens_kappa(reviewer_1, reviewer_2):.2f}")

> **Observe:** v2 resolves more conversations *and* catches more of the cases that need a human: better criteria move precision and recall together, unlike a looser or stricter threshold, which trades one for the other. And high raw agreement can hide a weak kappa when most items are "correct" anyway; if your labels are noisy, every reliability table built on them is noisy too.

## Mini-project: a support agent that hands off cleanly, plus a review plan

Two halves, combining the whole chapter:

1. **Support agent** (`get_customer`, `lookup_order`, `process_refund`, `escalate_to_human`) on the Messages API with:
   - the escalation policy prompt from 9.2,
   - the deterministic gate from 9.4 (verified-ID prerequisite, order ownership, $500 limit),
   - fail-closed auto-escalation when a blocked case is not escalated, and when the turn budget runs out,
   - a structured handoff validated by Pydantic, cross-checked against the gate state (`handoff_problems`), returned to the model as an `is_error` result when incomplete, and finalized by code before filing.
2. **Review plan** for the extraction pipeline: per-segment thresholds, which segments to automate, and an audit plan sized by the rule of three.

Run it offline first with scripted conversations, then live.

In [ ]:
SYSTEM = ESCALATION_POLICY + """

Operating rules:
- Verify the customer with get_customer before any order lookup or refund. If it returns multiple_matches,
  ask for one of the listed identifiers. Never pick an account yourself.
- When you escalate, fill every field of escalate_to_human. The specialist sees only that summary.
- After escalating, tell the customer what happens next without promising an outcome."""

TOOLS = SUPPORT_TOOLS + [escalate_to_human]


class SupportAgent:
    def __init__(self, llm: anthropic.Anthropic, max_turns: int = 8):
        self.llm, self.max_turns = llm, max_turns
        self.state = CaseState()
        self.events: list[str] = []

    def escalate(self, **summary) -> dict:
        try:
            h = HandoffSummary.model_validate(summary)
        except ValidationError as err:
            return {"status": "rejected", "problems": [f"{e['loc']}: {e['msg']}" for e in err.errors()]}
        problems = handoff_problems(h, self.state)
        if problems:
            return {"status": "rejected", "problems": problems}
        final = finalize_handoff(h, self.state)
        return file_ticket({"filed_by": "agent", **final.model_dump(), "gate_log": self.state.gate_log})

    def execute(self, block) -> dict:
        if block.name == "escalate_to_human":
            out = self.escalate(**block.input)
            if out["status"] == "rejected":
                self.events.append(f"handoff rejected: {out['problems']}")
                return {"type": "tool_result", "tool_use_id": block.id, "is_error": True,
                        "content": "Handoff incomplete, fix and call escalate_to_human again: " + "; ".join(out["problems"])}
            self.state.escalated = True
            self.events.append(f"escalated -> {out['ticket_id']}")
            return {"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(out)}
        result = execute(block, self.state, HANDLERS)
        self.events.append(f"{block.name}({json.dumps(block.input)[:50]}) -> {'DENY' if result.get('is_error') else 'ok'}")
        return result

    def run(self, user_text: str) -> str:
        messages = [{"role": "user", "content": user_text}]
        final = ""
        for _ in range(self.max_turns):
            r = self.llm.messages.create(model=MODEL, max_tokens=4096, system=SYSTEM, tools=TOOLS, messages=messages)
            messages.append({"role": "assistant", "content": r.content})
            if r.stop_reason == "tool_use":
                messages.append({"role": "user", "content": [self.execute(blk) for blk in r.content if blk.type == "tool_use"]})
                continue
            final = "".join(blk.text for blk in r.content if blk.type == "text") or f"[stopped: {r.stop_reason}]"
            break
        else:                                                   # turn budget exhausted: no progress
            final = "[turn budget exhausted]"
            if not self.state.escalated:
                self.events.append("auto-escalated: " + file_ticket({"filed_by": "harness", "escalation_reason": "cannot_make_progress",
                                                                     "customer_id": self.state.verified_customer_id})["ticket_id"])
                self.state.escalated = True
        ticket = close_case(self.state, final)                  # fail closed on blocked refunds
        if ticket:
            self.events.append(f"auto-escalated (fail-closed) -> {ticket['ticket_id']}")
        return final


def show_run(agent: SupportAgent, request: str) -> None:
    print("USER :", request)
    print("AGENT:", agent.run(request))
    for e in agent.events:
        print("   ·", e)
    print("   refunds:", [r["amount"] for r in REFUNDS], "| tickets:", [t["ticket_id"] + "/" + t["filed_by"] for t in TICKETS])
    print("-" * 100)

In [ ]:
def handoff(**overrides) -> dict:
    """A complete handoff for Maria (CUST-20002) and her fridge, with overrides for each scenario."""
    base = {"escalation_reason": "threshold_exceeded", "reason_detail": "Refund of $1,249 exceeds the $500 agent limit.",
            "customer_id": "CUST-20002", "identity_verified": True, "order_ids": ["ORD-70002"],
            "issue_summary": "Fridge stopped working a week after delivery; customer wants a full refund.",
            "root_cause": None, "facts_verified": ["ORD-70002 delivered 2026-09-21, paid $1,249.00"],
            "customer_claims": ["Fridge stopped cooling after one week"],
            "actions_taken": [{"step": "get_customer(email)", "outcome": "Single match, CUST-20002"},
                              {"step": "lookup_order(ORD-70002)", "outcome": "Delivered, $1,249.00"},
                              {"step": "process_refund(1249.00)", "outcome": "Blocked: above the $500 limit"}],
            "offers_made": [], "amount_at_stake_usd": 1249.0,
            "recommended_action": "Approve the full refund or send a technician within 48 hours.",
            "recommendation_rationale": "Failure within the first week is covered by warranty.",
            "urgency": "high", "open_questions": ["Refund or repair?"]}
    return {**base, **overrides}


reset_backend()
verify_maria = [reply(tool_call("get_customer", {"name": "Maria Lopez", "email": "mlopez@example.org", "phone": None})),
                reply(tool_call("lookup_order", {"order_id": "ORD-70002"}))]

scenarios = {
    "A. in scope, within the limit": ScriptedClaude(
        reply(tool_call("get_customer", {"name": None, "email": "ana@example.com", "phone": None})),
        reply(tool_call("lookup_order", {"order_id": "ORD-31544"})),
        reply(tool_call("process_refund", {"order_id": "ORD-31544", "amount": 179.99, "reason": "arrived broken"})),
        reply(text("Done: $179.99 is on its way back to your card (RF-0001)."))),
    "B. explicit request for a human": ScriptedClaude(
        reply(tool_call("escalate_to_human", handoff(escalation_reason="customer_requested_human",
              reason_detail="Customer asked for a manager in the first message.", customer_id=None,
              identity_verified=False, order_ids=[], actions_taken=[], facts_verified=[],
              amount_at_stake_usd=None, urgency="normal", open_questions=[]))),
        reply(text("I've passed your case to a team member; you won't need to repeat yourself."))),
    "C. above the limit, model escalates": ScriptedClaude(
        *verify_maria,
        reply(tool_call("process_refund", {"order_id": "ORD-70002", "amount": 1249.0, "reason": "dead fridge"})),
        reply(tool_call("escalate_to_human", handoff(customer_id="CUST-99999"))),     # wrong ID -> rejected
        reply(tool_call("escalate_to_human", handoff())),                              # fixed
        reply(text("A specialist will review the full refund within 4 business hours."))),
    "D. above the limit, model forgets to escalate": ScriptedClaude(
        *verify_maria,
        reply(tool_call("process_refund", {"order_id": "ORD-70002", "amount": 1249.0, "reason": "dead fridge"})),
        reply(text("Sorry, I can't refund that much."))),
    "E. three Marias: ask, don't guess": ScriptedClaude(
        reply(tool_call("get_customer", {"name": "Maria Lopez", "email": None, "phone": None})),
        reply(tool_call("process_refund", {"order_id": "ORD-70001", "amount": 89.99, "reason": "leaks"})),   # a guess
        reply(text("I found several accounts with that name. What's the email address on your account?"))),
}
requests_ = {"A": "I'm ana@example.com, my blender ORD-31544 arrived broken. Refund please.",
             "B": "I want to speak to a manager.",
             "C": "Maria Lopez, mlopez@example.org. My fridge ORD-70002 died, refund the $1,249.",
             "D": "Maria Lopez, mlopez@example.org. My fridge ORD-70002 died, refund the $1,249.",
             "E": "This is Maria Lopez, refund my kettle."}
for name, fake in scenarios.items():
    print(f"### {name}")
    show_run(SupportAgent(fake.client), requests_[name[0]])

> **Observe:**
> - **A** pays out because every rule passes. **B** escalates in the first turn without investigating; its handoff is valid even with `customer_id: null`, because identity was never verified and the gate state agrees.
> - **C** shows the validation loop: the first handoff names the wrong customer, comes back as an `is_error` result, and only the corrected one is filed. **D** shows fail-closed: the model gave up politely, and the harness filed the ticket anyway.
> - **E**: the model tried to refund on a guess, the gate denied it, and the model asked for an identifier instead.

In [ ]:
# (live) The same agent on the real model. Watch the events: which rules fired, and did the model follow the redirects?
if not HAS_KEY:
    print("skipped (live)")
else:
    reset_backend()
    for request in ["Hi, I'm ana@example.com. My blender (ORD-31544) arrived broken, can I get a refund?",
                    "I want to speak to a manager right now.",
                    "Maria Lopez here, mlopez@example.org. My fridge ORD-70002 died after a week. Refund the full $1,249.",
                    "This is Maria Lopez, please refund my kettle."]:
        show_run(SupportAgent(client), request)

### Part 2: a review plan for the extraction pipeline (offline)

Turn the labeled set into the plan you would hand to an operations lead: for each document type × field, automate or keep on review (calibrated threshold at a 98% target), the share of volume automated, and how many auto-accepted items to audit per stratum so that "zero errors found" would support a claim below 2%.

In [ ]:
def review_plan(records: list[dict], target: float = 0.98, min_n: int = 150, claim_below: float = 0.02) -> list[dict]:
    plan = []
    for doc_type in DOC_MIX:
        for f in FIELDS:
            seg = select(records, doc_type=doc_type, field=f)
            t = pick_threshold(seg, target, min_n)
            auto_share = 0.0 if t is None else sum(r["confidence"] >= t for r in seg) / len(seg)
            plan.append({"doc_type": doc_type, "field": f, "n": len(seg),
                         "accuracy": sum(r["correct"] for r in seg) / len(seg), "threshold": t,
                         "auto_share": auto_share, "monthly_audit": audits_needed(claim_below) if t is not None else 0})
    return plan


plan = review_plan(LABELED)
print(f"{'doc_type':20} {'field':13} {'n':>5} {'acc':>6} {'threshold':>9} {'auto':>6} {'audit/mo':>8}")
for row in plan:
    print(f"{row['doc_type']:20} {row['field']:13} {row['n']:5} {row['accuracy']:6.1%} "
          f"{('review' if row['threshold'] is None else row['threshold']):>9} {row['auto_share']:6.0%} {row['monthly_audit']:8}")
automated = [r for r in plan if r["threshold"] is not None]
print(f"\nsegments automated: {len(automated)}/{len(plan)}; audit load: {sum(r['monthly_audit'] for r in automated)} items/month; "
      "recalibrate on every model, prompt, schema or document-mix change")

> **Try it:**
> 1. Add a `max_refunds_per_case` rule to `gate()` and a scripted scenario that tries two refunds.
> 2. Route the agent's `escalate_to_human` tickets by `escalation_reason` into separate queues and print their sizes after the live run.
> 3. In the review plan, replace the fixed audit size with the rule of three per stratum *and* a minimum of 30, and add a column for the expected number of errors that slip through per month at production volume.

## Recap

- Escalate on an **explicit request**, a **policy gap or exception**, or **no progress**; code adds **business limits**. Sentiment and self-rated confidence misroute in both directions. Measure the policy on labeled turns: precision, recall, reason accuracy.
- The four patterns: immediate escalation, resolve first, acknowledge → offer → escalate on reiteration, policy-gap escalation. In the prompt: categorical criteria plus boundary examples.
- **Ask, don't guess** when the customer can resolve the ambiguity; tool results should make asking natural and leak nobody's data.
- Rules that must always hold go in **code**: prerequisite gates fed by tool results, denials with a next step, fail-closed escalation, single-use tokens for irreversible actions, and least privilege above all.
- Pause for humans by surface: persist-and-resume on the Messages API, `can_use_tool` (only for calls nothing else approved), `PreToolUse` → `defer` for long waits, Managed Agents `always_ask` (custom tools are yours to gate).
- Handoffs are the `escalate_to_human` schema: verified facts vs claims, actions tried, amount, recommendation, open questions; validated and finalized by code.
- Confidence is a ranking until calibrated: per-field, per-segment thresholds from labeled data, then stratified random audits of the auto-accepted stream, routing by P(error) × cost, and escalation metrics on every change.

Next: [homework](02_homework.ipynb)